# Notebook 21 — Signed-Motif Gene Overlay

## The follow-up question

Nb03b found that gene expression at class level does **not** predict **unsigned** motif participation (0/60k FDR). Nb13 found that the **signed** connectome has real structure (6/8 sign classes significant, with (−,+,−) at z=+28).

**Could gene expression predict participation in specific *signed* motif classes, even though it failed for unsigned motifs?** The biological rationale: a (+,+,+) coherent amplifier and a (−,−,−) mutual silencer look identical as unsigned triangles, but have opposite functions and likely different gene signatures. By collapsing signs, Nb03b may have averaged over heterogeneous biology.

## What we're doing

For each neuron, compute participation counts in each of the three strongly-enriched signed motif classes from Nb13:
- **(+,+,+)** coherent FFL (z = +3.6, enriched)
- **(−,+,−)** dual-inhibition gate (z = +28, massively enriched)
- *(optional)* The depleted mixed-sign classes as negative controls

Then run class-level Spearman of per-class gene expression against per-class motif counts, with BH-FDR across all tests.

## Preregistered criteria

1. **Per-class signed-motif counts are non-zero** for at least 30 classes per motif type (need variance for correlation).
2. **At least one gene survives q_global < 0.05** for either of the two target motifs — critically **better than Nb03b's 0/60k**.
3. **Permutation null (shuffle class labels)**: real FDR-survivor count > null 95pct.

## Halting rule

If criterion 2 fails: signed motifs do not recover a gene signal at class level either. Consistent with Nb03b — genes don't predict per-class motif participation at this sample size regardless of sign information.

If criterion 2 passes: we have a gene-level result that was missed by the unsigned approach — first such finding in the project for class-level motif-gene correlation.

In [1]:
import sys, time
from pathlib import Path
import warnings; warnings.simplefilter('ignore')
_HERE = Path.cwd()
if (_HERE / 'lib').is_dir(): sys.path.insert(0, str(_HERE))
elif (_HERE.parent / 'lib').is_dir(): sys.path.insert(0, str(_HERE.parent))

from lib.paths import DERIVED
from lib.reference import load_nt_reference
from lib.lr_compatibility import load_lr_atlas

import numpy as np, pandas as pd
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

RNG = np.random.default_rng(42)

# Adult connectome + NT + LR
adult = np.load(DERIVED / 'connectome_adult.npz', allow_pickle=True)
w_neurons = np.array([str(n) for n in adult['neurons']])
w_chem = adult['chem_adj']

mapping = pd.read_csv(DERIVED / 'expression_neuron_mapping.csv')
neuron_to_class = dict(zip(mapping['witvliet_name'], mapping['cengen_class']))

nt = load_nt_reference()
lr = load_lr_atlas()

expr_data = np.load(DERIVED / 'expression_tpm.npz', allow_pickle=True)
tpm_neurons = expr_data['neurons']
tpm = expr_data['tpm']
genes_wbg = expr_data['genes_wbg']
genes_csv = pd.read_csv(DERIVED / 'expression_genes.csv')
gene_symbols = genes_csv['symbol'].values

print(f'Witvliet: {len(w_neurons)} neurons, {int((w_chem>0).sum())} chem edges')

Witvliet: 222 neurons, 2191 chem edges


## Step 1 — Reconstruct signed edges (from Nb13)

In [2]:
# Class-level CeNGEN expression for receptor lookups
class_to_expr = {}
for i, nm in enumerate(tpm_neurons):
    cls = neuron_to_class.get(str(nm))
    if isinstance(cls, str) and cls not in class_to_expr and not np.all(np.isnan(tpm[i])):
        class_to_expr[cls] = tpm[i]

sym_to_idx = {s: i for i, s in enumerate(gene_symbols)}

EXCIT_GENES = ['acr-2','acr-5','acr-8','acr-10','acr-12','acr-14','acr-15','acr-16','acr-17','acr-18','acr-19','acr-20','acr-21','acr-23','acr-24','acr-25',
               'unc-29','unc-38','unc-63','lev-1','lev-8','des-2',
               'glr-1','glr-2','glr-3','glr-4','glr-5','glr-6','glr-7','glr-8','nmr-1','nmr-2','avr-14','avr-15']
INHIB_GENES = ['unc-49','gab-1','lgc-37','gbb-1','gbb-2',
               'glc-1','glc-2','glc-3','glc-4',
               'gar-2','gar-3']

TPM_THRESH = 10.0

def class_expresses(cls, gene_list):
    if cls not in class_to_expr: return set()
    vec = class_to_expr[cls]
    return {g for g in gene_list if (sym_to_idx.get(g) is not None) and (vec[sym_to_idx[g]] >= TPM_THRESH)}

def get_nt(neuron):
    v = nt.nt_of(neuron)
    if v is None: return None
    s = v.lower()
    if 'acetylcholine' in s: return 'ACh'
    if 'gaba' in s: return 'GABA'
    if 'glutamate' in s: return 'Glu'
    return None

def edge_sign(pre, post):
    nt_p = get_nt(pre)
    if nt_p is None: return 0
    cls_q = neuron_to_class.get(post)
    if not isinstance(cls_q, str): return 0
    ex = class_expresses(cls_q, EXCIT_GENES)
    ih = class_expresses(cls_q, INHIB_GENES)
    if nt_p == 'ACh':
        nic = {g for g in ex if g.startswith('acr-') or g in ['unc-29','unc-38','unc-63','lev-1','lev-8','des-2']}
        musc = {g for g in ih if g.startswith('gar-')}
        if nic and not musc: return +1
        if musc and not nic: return -1
        if nic: return +1
        return 0
    if nt_p == 'GABA':
        if any(g in ih for g in ['unc-49','gab-1','lgc-37','gbb-1','gbb-2']): return -1
        return 0
    if nt_p == 'Glu':
        iglu = {g for g in ex if g.startswith('glr-') or g.startswith('nmr-') or g.startswith('avr-')}
        glc  = {g for g in ih if g.startswith('glc-')}
        if iglu and not glc: return +1
        if glc and not iglu: return -1
        if iglu: return +1
        return 0
    return 0

N = len(w_neurons)
signs = np.zeros((N, N), dtype=np.int8)
for i in range(N):
    for j in range(N):
        if w_chem[i, j] > 0:
            signs[i, j] = edge_sign(w_neurons[i], w_neurons[j])
n_pos = int((signs > 0).sum())
n_neg = int((signs < 0).sum())
print(f'Signed edges: +{n_pos}, -{n_neg}, unassigned={int((w_chem>0).sum()) - n_pos - n_neg}')

Signed edges: +1350, -39, unassigned=802


## Step 2 — Per-neuron signed motif participation

In [3]:
# For each neuron i, count participation in:
#   coherent_ffl (+,+,+): i->j, j->k, i->k all excitatory, i as ANY anchor
#   dual_inhibition (-,+,-): i->j(inh), j->k(exc), i->k(inh) — count as i-anchor where i is the inhibitor
# Motif participation per node i: number of motifs involving i (as any role).

A_pos = (signs > 0).astype(np.int32)
A_neg = (signs < 0).astype(np.int32)

# FFL with sign class: triples (i, j, k) with:
#   i->j of sign s_ij, j->k of sign s_jk, i->k of sign s_ik
# Participation count for neuron u = # of ordered (j, k) tuples where u == i (anchor)
#                                  + # where u == j (middle)
#                                  + # where u == k (output)
# We care about total participation.

def signed_ffl_participation(s_ij, s_jk, s_ik):
    A_ij = A_pos if s_ij == +1 else A_neg
    A_jk = A_pos if s_jk == +1 else A_neg
    A_ik = A_pos if s_ik == +1 else A_neg
    # Count of triangles with this sign pattern anchored at each i:
    # For each i: sum over j,k of A_ij[i,j] * A_jk[j,k] * A_ik[i,k]
    # = for each i: (A_ij @ A_jk) * A_ik  then sum along k axis → gives count of triangles with i as anchor
    A2 = A_ij @ A_jk
    per_i_anchor = np.sum(A2 * A_ik, axis=1)    # neuron i is the source of i->j and i->k
    per_j_middle = np.sum(A_ij * A2, axis=0)    # neuron j is the middle
    per_k_output = np.sum(A_jk * A_ik, axis=1)  # actually need to transpose — skip, use per_i as proxy
    # Total participation: simpler aggregate — for each neuron u, count triangles touching u in any role.
    total = per_i_anchor + per_j_middle + np.sum(A2 * A_ik, axis=0)  # approximate
    return total, per_i_anchor

# Participation in (+,+,+):
plus_triple_total, plus_anchor = signed_ffl_participation(+1, +1, +1)
# Participation in (-,+,-):
minus_plus_minus_total, dualinh_anchor = signed_ffl_participation(-1, +1, -1)

# For simplicity, use per_i_anchor as the target (clearer role interpretation)
print(f'(+,+,+) anchor count per-neuron: mean={plus_anchor.mean():.2f}, nonzero={int((plus_anchor>0).sum())}')
print(f'(-,+,-) anchor count per-neuron: mean={dualinh_anchor.mean():.2f}, nonzero={int((dualinh_anchor>0).sum())}')

(+,+,+) anchor count per-neuron: mean=10.16, nonzero=120
(-,+,-) anchor count per-neuron: mean=0.17, nonzero=1


## Step 3 — Collapse to class level and run gene × signed-motif tests

In [4]:
# Aggregate per-neuron anchor counts to per-class means
per_neuron = pd.DataFrame({
    'neuron': w_neurons,
    'class': [neuron_to_class.get(str(n)) for n in w_neurons],
    'plus_anchor': plus_anchor,
    'dualinh_anchor': dualinh_anchor,
})
per_class = per_neuron.dropna(subset=['class']).groupby('class')[['plus_anchor','dualinh_anchor']].mean()
classes_with_data = sorted(class_to_expr.keys())
common_classes = [c for c in classes_with_data if c in per_class.index]
per_class_aligned = per_class.loc[common_classes]

# Gene expression matrix at class level, filtered by variance (as in Nb03b)
X_cls = np.stack([class_to_expr[c] for c in common_classes])
n_expressed = (X_cls > 0).sum(axis=0)
means = X_cls.mean(axis=0); stds = X_cls.std(axis=0)
cv = np.where(means > 0, stds / (means + 1e-9), 0.0)
keep = (n_expressed >= 5) & (cv >= 1.0)
print(f'N classes: {len(common_classes)}  Genes after filter: {int(keep.sum())}')

X_cls_filt = X_cls[:, keep]
wbg_filt = genes_wbg[keep]
sym_filt = gene_symbols[keep]

# Check variance of targets
for m in ['plus_anchor', 'dualinh_anchor']:
    y_vals = per_class_aligned[m].values
    print(f'{m}: mean={y_vals.mean():.2f}, std={y_vals.std():.2f}, nonzero={int((y_vals>0).sum())}')

N classes: 84  Genes after filter: 8846
plus_anchor: mean=14.46, std=19.00, nonzero=55
dualinh_anchor: mean=0.44, std=4.01, nonzero=1


In [5]:
MOTIF_TARGETS = ['plus_anchor', 'dualinh_anchor']
G = X_cls_filt.shape[1]
rows = []
t0 = time.time()
for target in MOTIF_TARGETS:
    y_vals = per_class_aligned[target].values.astype(float)
    for gi in range(G):
        x = X_cls_filt[:, gi]
        r, p = spearmanr(x, y_vals)
        rows.append({'motif': target, 'wbgene': wbg_filt[gi], 'symbol': sym_filt[gi],
                     'rho': float(r) if not np.isnan(r) else 0.0, 'p': float(p) if not np.isnan(p) else 1.0})
print(f'Done in {time.time()-t0:.1f}s ({len(rows)} tests)')
results = pd.DataFrame(rows)

results['q_global'] = multipletests(results['p'].fillna(1.0).values, method='fdr_bh')[1]
results['q_per_motif'] = np.nan
for m in MOTIF_TARGETS:
    mask = results['motif'] == m
    results.loc[mask, 'q_per_motif'] = multipletests(results.loc[mask, 'p'].fillna(1.0).values, method='fdr_bh')[1]
results = results.sort_values('p').reset_index(drop=True)

print(f'\nTop 15 gene × signed-motif hits:')
print(results.head(15).to_string())

n_global_fdr = int((results['q_global'] < 0.05).sum())
n_per_motif_fdr = int((results['q_per_motif'] < 0.05).sum())
print(f'\nSurvivors at q_global < 0.05:    {n_global_fdr}')
print(f'Survivors at q_per_motif < 0.05: {n_per_motif_fdr}')

by_motif = results[results['q_per_motif'] < 0.05].groupby('motif').size()
print(f'\nPer-motif FDR hits: {by_motif.to_dict()}')

results.to_csv(DERIVED / 'nb21_gene_signed_motif.csv', index=False)

Done in 3.8s (17692 tests)

Top 15 gene × signed-motif hits:
             motif          wbgene      symbol       rho         p  q_global  q_per_motif
0   dualinh_anchor  WBGene00022217    fbxa-221  0.458120  0.000012  0.032384     0.016192
1   dualinh_anchor  WBGene00003999       pgp-5  0.458120  0.000012  0.032384     0.016192
2   dualinh_anchor  WBGene00002073       ima-2  0.458120  0.000012  0.032384     0.016192
3   dualinh_anchor  WBGene00007916     C34C6.3  0.458120  0.000012  0.032384     0.016192
4   dualinh_anchor  WBGene00235286    C01F6.16  0.458120  0.000012  0.032384     0.016192
5   dualinh_anchor  WBGene00021835  Y54E10A.17  0.447081  0.000020  0.032384     0.016192
6   dualinh_anchor  WBGene00000521       cky-1  0.447081  0.000020  0.032384     0.016192
7   dualinh_anchor  WBGene00011822     T18D3.5  0.447081  0.000020  0.032384     0.016192
8   dualinh_anchor  WBGene00185005    T03F6.10  0.447081  0.000020  0.032384     0.016192
9   dualinh_anchor  WBGene00269438   Y5

## Step 4 — Permutation null

In [6]:
N_PERM = 30
null_hits = []
t0 = time.time()
for p in range(N_PERM):
    perm_classes = RNG.permutation(common_classes)
    per_class_shuf = per_class_aligned.reset_index(drop=True)
    per_class_shuf.index = perm_classes
    per_class_shuf = per_class_shuf.loc[common_classes]
    X_shuf = X_cls_filt  # keep genes aligned to original class order
    # Actually the permutation should shuffle y values, not class labels
    # Simpler:
    total_hits = 0
    for target in MOTIF_TARGETS:
        y_vals = per_class_aligned[target].values.astype(float)
        y_perm = RNG.permutation(y_vals)
        pvals = []
        for gi in range(X_shuf.shape[1]):
            _, pv = spearmanr(X_shuf[:, gi], y_perm)
            pvals.append(pv if not np.isnan(pv) else 1.0)
        q = multipletests(pvals, method='fdr_bh')[1]
        total_hits += int((q < 0.05).sum())
    null_hits.append(total_hits)
    if (p+1) % 10 == 0:
        print(f'  {p+1}/{N_PERM}  ({time.time()-t0:.1f}s)  latest: {null_hits[-1]} null hits')
null_hits = np.array(null_hits)
print(f'\nNull distribution of gene-signed-motif FDR hits:')
print(f'  mean: {null_hits.mean():.1f}, 95pct: {np.percentile(null_hits, 95):.1f}, max: {null_hits.max()}')
print(f'Real: {n_per_motif_fdr} (per-motif FDR hits)')

  10/30  (36.2s)  latest: 87 null hits


  20/30  (71.9s)  latest: 29 null hits


  30/30  (107.8s)  latest: 8 null hits

Null distribution of gene-signed-motif FDR hits:
  mean: 50.7, 95pct: 114.0, max: 160
Real: 44 (per-motif FDR hits)


## Step 5 — Criteria

In [7]:
crit1_plus = int((per_class_aligned['plus_anchor'] > 0).sum()) >= 30
crit1_dual = int((per_class_aligned['dualinh_anchor'] > 0).sum()) >= 30
crit1 = crit1_plus and crit1_dual
crit2 = n_per_motif_fdr >= 1
crit3 = n_per_motif_fdr > np.percentile(null_hits, 95)

print('CRITERIA')
print('=' * 70)
print(f'  [{"PASS" if crit1 else "FAIL"}] 1 >= 30 classes with nonzero counts in both motifs')
print(f'       (+,+,+): {int((per_class_aligned["plus_anchor"] > 0).sum())} classes')
print(f'       (-,+,-): {int((per_class_aligned["dualinh_anchor"] > 0).sum())} classes')
print(f'  [{"PASS" if crit2 else "FAIL"}] 2 >= 1 gene at q_per_motif < 0.05         {n_per_motif_fdr}')
print(f'  [{"PASS" if crit3 else "FAIL"}] 3 Real hits > null 95pct                   real={n_per_motif_fdr}, null={np.percentile(null_hits, 95):.1f}')
print('=' * 70)

if crit1 and crit2 and crit3:
    verdict = 'POSITIVE — gene expression predicts signed-motif participation (missed by unsigned Nb03b)'
elif crit1 and crit2:
    verdict = 'POSITIVE (FDR) but not above null — some signal but hard to separate from noise'
elif crit1:
    verdict = 'NULL — even signed motifs don\'t recover a gene signal at N=84'
else:
    verdict = 'INCONCLUSIVE — insufficient motif coverage for test'
print(f'\nVERDICT: {verdict}')

summary = pd.DataFrame([{
    'n_classes': len(common_classes),
    'n_plus_nonzero': int((per_class_aligned['plus_anchor'] > 0).sum()),
    'n_dualinh_nonzero': int((per_class_aligned['dualinh_anchor'] > 0).sum()),
    'n_tests': len(results),
    'n_fdr_global': n_global_fdr,
    'n_fdr_per_motif': n_per_motif_fdr,
    'null_95pct': float(np.percentile(null_hits, 95)),
    'verdict': verdict,
}])
summary.to_csv(DERIVED / 'nb21_final_summary.csv', index=False)
print(summary.T.to_string())

CRITERIA
  [FAIL] 1 >= 30 classes with nonzero counts in both motifs
       (+,+,+): 55 classes
       (-,+,-): 1 classes
  [PASS] 2 >= 1 gene at q_per_motif < 0.05         44
  [FAIL] 3 Real hits > null 95pct                   real=44, null=114.0

VERDICT: INCONCLUSIVE — insufficient motif coverage for test
                                                                     0
n_classes                                                           84
n_plus_nonzero                                                      55
n_dualinh_nonzero                                                    1
n_tests                                                          17692
n_fdr_global                                                        25
n_fdr_per_motif                                                     44
null_95pct                                                       114.0
verdict            INCONCLUSIVE — insufficient motif coverage for test
